# Phase 0 — Base Foundation Check

**RBI-ObliBench / Agentic RAG compliance system**

Verifies that the shared foundation on the `base` branch imports, configures,
and runs correctly **inside the Kaggle environment**, which is where all
compute for this project happens.

This notebook is a thin wrapper. Every cell imports and calls code in `src/`;
no logic lives here. Run all cells top to bottom.

Requires **Internet: On** in notebook settings (to clone the repository).

## 1. Get the code

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/karanLokhande29/Capstone_project.git"
BRANCH = "base"

# /kaggle/working is the only writable location on Kaggle; fall back locally.
WORKING = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
REPO_DIR = os.path.join(WORKING, "Capstone_project")

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, REPO_DIR],
        check=True,
    )

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("repository:", REPO_DIR)
print("branch:", subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"],
                               capture_output=True, text=True).stdout.strip())

## 2. Dependencies

Phase 0 needs only the standard library plus PyYAML, both already in Kaggle's
base image, so **no pip install is required**. The cell below reports what is
actually present rather than assuming `requirements.txt` was right.

In [ ]:
from src.common.verify import environment_versions

for name, version in environment_versions().items():
    print(f"{name:12s} {version}")

## 3. Unit tests

232 tests covering the config loader, dual-mode path resolution, I/O helpers,
retry/backoff, the cache, and every shared schema. No network calls.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-q"],
    capture_output=True, text=True,
)
print(result.stdout[-4000:])
if result.returncode != 0:
    print(result.stderr[-4000:])
print("UNIT TESTS:", "PASS" if result.returncode == 0 else "FAIL")

## 4. Integration smoke test

Imports every module, checks for circular imports and missing `__init__.py`,
loads the config, resolves every declared path key **in this environment**,
and round-trips the cache. This is the check that could not be run locally,
because locally there is no `/kaggle/input` or `/kaggle/working`.

In [ ]:
from src.common.verify import format_report, run_foundation_check

report = run_foundation_check()
print(format_report(report))

## 5. Environment actually detected

Confirms the dual-mode resolver identified Kaggle and picked the right roots:
writes to `/kaggle/working`, reads from attached Datasets first.

In [ ]:
from src.common.config import load_config
from src.common.paths import PathResolver

cfg = load_config()
paths = PathResolver.from_config(cfg)

print("mode:         ", paths.mode.value)
print("writes go to: ", paths.working_root)
print("reads search: ", [str(p) for p in paths.search_roots()])

assert paths.mode.value == "kaggle", (
    "Expected Kaggle mode. If this fails on Kaggle, the environment detection "
    "in src/common/paths.py needs attention."
)

## 6. Shared schema contracts

The field tables Phase 1 branches build against, rendered from the code itself
so this output cannot drift from the contract. The **Populated by** column is
the machine-readable answer to whose job each field is.

In [ ]:
from IPython.display import Markdown, display

from src.schemas import ALL_SCHEMAS

for schema in ALL_SCHEMAS:
    display(Markdown(f"### {schema.__name__}\n\n{schema.__doc__.strip().splitlines()[0]}"))
    display(Markdown(schema.spec_table()))

## 7. Result

Copy the line below into the weekly logbook.

In [ ]:
summary = report["summary"]
print(f"Integration smoke test: {report['overall']} "
      f"({summary['checks_passed']}/{summary['checks_run']} checks)")
print(f"Unit tests: {'PASS' if result.returncode == 0 else 'FAIL'}")
print(f"Kaggle notebook: phase0-base-foundation")

---

### Saving output as a Kaggle Dataset

Nothing in Phase 0 produces data worth persisting, so no dataset is needed yet.

From Phase 1 onward, when a run produces artifacts to keep:

1. Confirm the files are under `/kaggle/working/` (they will be — `PathResolver`
   writes nowhere else).
2. Use **New Dataset** in the Kaggle UI, or **New Version** on an existing one.
3. Add the dataset slug to `environment.kaggle.input_datasets` in
   `config/config.yaml` and commit it, so the next session reads those files as
   read-only input — and the cache treats them as hits instead of re-fetching.